In [ ]:
%load_ext autoreload
%autoreload 2

Declaration of parameters (you must also add a tag for this cell - parameters)

In [ ]:
# specify substep parameters for interactive run
# this cell will be replaced during job run with the parameters from json within params subfolder
substep_params={   
    "min_score" : 0.05
}

In [ ]:
# load pipeline and step parameters - do not edit
from sinara.substep import get_pipeline_params, get_step_params
pipeline_params = get_pipeline_params(pprint=True)
step_params = get_step_params(pprint=True)

In [ ]:
# define substep interface
from sinara.substep import NotebookSubstep, ENV_NAME, PIPELINE_NAME, ZONE_NAME, STEP_NAME, RUN_ID, ENTITY_NAME, ENTITY_PATH, SUBSTEP_NAME

substep = NotebookSubstep(pipeline_params, step_params, substep_params)

substep.interface(
    inputs =    
    [ 
      { STEP_NAME: "data_prep", ENTITY_NAME: "test_data"}, # dataset for detector testing from data_prep step
      { STEP_NAME: "model_pack", ENTITY_NAME: "bento_service"}  # stored BentoService from model_pack
    ],
    tmp_entities =
    [
        { ENTITY_NAME: "obj_detect_inference_files" }
    ],
    tmp_outputs =
    [
        { ENTITY_NAME: "test_data" }, # temporary test dataset files for use in next substep
        { ENTITY_NAME: "predict_data" } 
    ],
)

substep.print_interface_info()

substep.exit_in_visualize_mode()

In [ ]:
# run spark
from sinara.spark import SinaraSpark

spark = SinaraSpark.run_session(0)
SinaraSpark.ui_url()

### Loading a trained model from bento_service
(weights, configs)

#### Get and load bentoservice

In [ ]:
import os.path as osp
import os
from pathlib import Path
from sinara.bentoml import load_bentoservice

# read trained model
print('read trained model')
inputs_model_pack = substep.inputs(step_name = "model_pack")
bento_service = load_bentoservice(inputs_model_pack.bento_service)

#### Unpacking and saving artifacts from bentoservice

In [ ]:
# Save binary model weights and config to tmp_entities.obj_detect_inference_files
from sinara.bentoml import save_bentoartifact_to_tmp

tmp_entities = substep.tmp_entities()
tmp_outputs = substep.tmp_outputs()

#model_name = bento_service.artifacts["model_name"].get()
model_path = osp.join(tmp_entities.obj_detect_inference_files, f"latest.pth")
cfg_path = osp.join(tmp_entities.obj_detect_inference_files, f"last_cfg.py")

save_bentoartifact_to_tmp(bento_service, artifact_name="model", artifact_file_path=model_path)
save_bentoartifact_to_tmp(bento_service, artifact_name="config", artifact_file_path=cfg_path)

### Loading test datasets (from step data_prep)

In [ ]:
from sinara.archive import SinaraArchive
import json

# copy test data and dataset_config from previos step (data_prep) to tmp
inputs_data_prep = substep.inputs(step_name = "data_prep")

archive = SinaraArchive(spark)
archive.unpack_files_from_store_to_tmp(store_path=inputs_data_prep.test_data, tmp_dir=tmp_outputs.test_data)

# reading ground-true test dataset markup 
with open(osp.join(tmp_outputs.test_data, "test_coco_annotations.json")) as f_id:
    eval_gt = json.load(f_id)
    
print('Amount of eval_gt annotations:', len(eval_gt['annotations']))
print('Keys of eval_gt:', list(eval_gt))

### Inference model

#### Initializing modules from mmdetection, mmcv

In [ ]:
from mmdet.apis import init_detector, inference_detector

#### Initializing model

In [ ]:
# build the model from a config file and a checkpoint file
model = init_detector(osp.join(tmp_entities.obj_detect_inference_files, 'last_cfg.py'), 
                      osp.join(tmp_entities.obj_detect_inference_files, 'latest.pth'), 
                      device='cpu')

#### Inference model on a test dataset

##### Optimization of model predictor in coco json format

In [ ]:
loaded_categories = [{'id': (i+1), 'name': _cat_name} for i, _cat_name in enumerate(model.dataset_meta["classes"])]
loaded_categories_remap = {i: _cat['id'] for i, _cat in enumerate(loaded_categories)}


In [ ]:
def map_result_as_dict(selected_classes,  
                       selected_cnts,
                       selected_boxes,
                       selected_scores,                        
                       output_shape: tuple, 
                       file_name: str = '', 
                       image_id: int = 0, 
                       min_score: float = 0) -> dict:
        """
            Optimized universal predictor
            :param selected_classes - numpy array of classes (-1,1)
            :param selected_cnts    - numpy array of contours (-1,-1,1,2)
            :param selected_boxes   - numpy array bbox (-1,4)
            :param selected_scores   - numpy array bbox (-1,1)
            :param output_shape     - output image shape. Usung only 2 first values
            :param file_name        - input file name
            :param use_segm         - using segmentation model or object detection only
            :param image_id         - input file id
            :param min_score        - filtering condition

            :return: dict
        """

        annotations = [{
            'id': i,
            'base_id': i,
            'image_id': int(image_id),
            # bbox x1,y1,w,h describing the object
            'bbox': [int(x1), int(y1), int(x2-x1), int(y2-y1)],
            # Object category
            'category_id': loaded_categories_remap[int(selected_classes[i])],
            'score': float(selected_scores[i]),  # confidence
            # Contour array x,y,x,y
            'segmentation': [np.int0(_cnt).ravel().tolist() for _cnt in selected_cnts[i]],
        } for i, ((x1, y1, x2, y2), score) in enumerate(zip(selected_boxes,selected_scores)) if score > min_score]

        for i in range(len(annotations)):
            del annotations[i]['base_id']

        return {
            'file_name': file_name,  # not used
            'height': output_shape[0],  # Image Resolution
            'width': output_shape[1],  # Image Resolution
            'image_id': int(image_id),
            'CLASSES_COUNT': len(loaded_categories),
            'annotations': annotations,
        }

##### Inference models on a test dataset and conversion to predicts in coco json format

In [ ]:
from tqdm import tqdm
import cv2
import numpy as np

eval_dt = dict(eval_gt, **{"annotations" : []})
max_out = None
img_prefix = tmp_outputs.test_data
for image in tqdm(eval_gt['images'], desc='process predict'):
    image_id  = image['id']
    file_name = osp.join(img_prefix, image['file_name'])
    img = cv2.imread(file_name)
    out = inference_detector(model, [file_name])    
    out_coco  = []
    for batch_i, out_img in enumerate(out):
        result = out_img.cpu().numpy().pred_instances
        
        selected_classes = []
        selected_boxes = []
        selected_scores = []
        selected_cnts = []
        
        if result.bboxes.size > 0:
            selected_scores = result.scores
            selected_boxes = result.bboxes
            selected_classes = result.labels
        
        box_to_cnt_indexes = [[0, 1], [2, 1], [2, 3], [0, 3], [0, 1]]
        for bbox in selected_boxes:
            cnt = []
            for a, b in box_to_cnt_indexes:
                cnt.append([bbox[a], bbox[b]])
            selected_cnts.append([cnt])
        
        out_coco.append(map_result_as_dict(
                        selected_classes,
                        selected_cnts,
                        selected_boxes,
                        selected_scores, 
                        img.shape[:2],
                        file_name=file_name,
                        image_id=image_id,
                        min_score=substep_params['min_score'],))
        
        eval_dt["annotations"] += out_coco[0].get("annotations", [])
        
print('eval_gt annotations amount:', len(eval_gt['annotations']))
print('eval_dt annotations amount', len(eval_dt['annotations']))

eval_dt['info']['iouType'] = 'bbox'
for ann in eval_dt['annotations']:
    ann['area'] = ann['bbox'][2] * ann['bbox'][3]
    
from utils.coco import dump as dump_coco

dump_coco(osp.join(tmp_outputs.predict_data, 'eval_dt_torch.json'), eval_dt)
print(f"eval_dt_torch = {osp.join(tmp_outputs.predict_data, 'eval_dt_torch.json')}")

### Preview of model prediction on an image

#### Build visualizer

In [ ]:
from mmengine.registry import VISUALIZERS
from typing import Optional

try:
    @VISUALIZERS.register_module()
    class DetLocalVisualizer(Visualizer):
        def add_datasample(self,
                           name,
                           image: np.ndarray,
                           data_sample: Optional['BaseDataElement'] = None,
                           draw_gt: bool = False,
                           draw_pred: bool = True,
                           show: bool = False,
                           wait_time: int = 0,
                           step: int = 0) -> None:
            pass
except Exception as e:
    print(e.__str__)

visualizer_cfg = dict(type='DetLocalVisualizer',
                      vis_backends=[dict(type='LocalVisBackend')],
                      name='visualizer')

# global initialize
VISUALIZERS.build(visualizer_cfg)

#### Visualizate last predict to last image from testin dataset

In [ ]:
from mmengine.visualization import Visualizer
import mmcv
# get built visualizer
visualizer_now = Visualizer.get_current_instance()
# the dataset_meta is loaded from the checkpoint and
# then pass to the model in init_detector
visualizer_now.dataset_meta = model.dataset_meta
# show the results
image = mmcv.imread(file_name, channel_order='rgb')
visualizer_now.add_datasample(
    'new_result',
    image,
    data_sample= out[0],
    draw_gt=False,
    wait_time=0,
    out_file=None,
    pred_score_thr=0.01
)
visualizer_now.show()

In [ ]:
# stop spark
SinaraSpark.stop_session()